**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/clark-2026"], check=True)
    os.chdir("/content/encore/entries/clark-2026")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Clark, Abbott & Sompolinsky (2026): the mean-field theory and the large-N spectra

Paper. doi:10.1038/s41593-026-02452-4; claims file `claims/clark-2026.yaml`. Dataset.
DANDI:000939 at version 0.240528.1542, the version the paper cites (see
`tuning_curves.ipynb`).

These claims are on the paper's theory and simulations (`data: elsewhere`, the paper's code at
commit 245726f), except `fig7c-dmft-integrates-behavior`, which drives the mean-field equations
with the head direction of mouse A3707 from the dataset. The mean-field code is a copy of the
paper's `ring/dmft.py` (`ring_dmft.py` in this entry, MIT licence, imports of torch removed).

| claim | data | tested here |
|---|---|---|
| fig7-dmft-mexican-hat-and-stability | elsewhere | yes |
| fig7de-dmft-timescales | elsewhere | yes |
| fig7c-dmft-integrates-behavior | dataset | yes |
| fig6cd-eigenvalues-real-degenerate | elsewhere | yes |
| fig6fg-disordered-eigenvectors | elsewhere | yes |

In [ ]:
import time, resource
import numpy as np
import pandas as pd
import scipy.linalg
from scipy.special import erf
from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import interp1d
from astropy.stats import circcorrcoef
import hdring as H
import ring_dmft as D

T0 = time.time()

## Data: the head direction of A3707

The session with most head-direction cells (`closing_the_loop.ipynb` picks it by count; A3707,
117 cells). Its square-epoch head direction is prepared as in `data_driven.preprocess_hd_data`.

In [ ]:
order = H.session_order(H.list_assets())
aid = [o[1] for o in order if o[2] == "A3707"][0]
s7 = H.load_session(aid)
print("A3707: units", s7["n_units"], "| head-direction units", int(s7["is_hd"].sum()), "| epochs", s7["epochs"])
ti_nom, tf_nom = [(a, b) for tg, a, b in s7["epochs"] if tg == "wake_square"][0]
ti_idx = np.argmin((ti_nom - s7["hd_t"]) ** 2); tf_idx = np.argmin((tf_nom - s7["hd_t"]) ** 2)
hd_ts = s7["hd"][ti_idx:tf_idx].copy()
print(f"square epoch {s7['hd_t'][ti_idx]:.2f} to {s7['hd_t'][tf_idx]:.2f} s; {len(hd_ts)} samples at median step {np.median(np.diff(s7['hd_t'][ti_idx:tf_idx]))*1e3:.2f} ms; "
      f"untracked {np.isnan(hd_ts).sum()}; range {np.nanmin(hd_ts):.4f} to {np.nanmax(hd_ts):.4f} rad ({s7['hd_unit']})")
def interpolate_angle_nans(hd):
    ok = np.where(~np.isnan(hd))[0]
    if len(ok) in (0, len(hd)): return hd
    zz = np.exp(1j * hd[ok]); ia = np.arange(len(hd))
    return np.mod(np.angle(np.interp(ia, ok, zz.real) + 1j * np.interp(ia, ok, zz.imag)), 2 * np.pi)
hu = np.unwrap(interpolate_angle_nans(hd_ts))
hu = np.concatenate((np.ones(2000) * hu[0], hu))
omega_data = np.diff(np.unwrap(gaussian_filter1d(hu, sigma=10))) / 0.01
print(f"omega: {len(omega_data)} samples, range {omega_data.min():.2f} to {omega_data.max():.2f} rad/s, zero for the first 20 s: {np.allclose(omega_data[:1990], 0, atol=1e-3)}")

## fig7-dmft-mexican-hat-and-stability

> In the mean-field theory the effective interactions are circulant with a Mexican hat profile
> set by sigma and beta; at a bump state all Jacobian eigenvalues have negative real parts except
> the symmetry-induced zero, the uniform state has a real positive eigenvalue for all sigma and
> beta (a Turing instability), and fixing Q destabilizes the bump.

As `dmft.ipynb`: the profiles of Fig. 7a (`plot_weights`, N_theta 1,000, sigma 1, 2, 4, beta
1.5, 3, 5, 10) and the stability grid (`run_analysis`, beta logspace(-2, 2, 40), sigma
logspace(0, 2, 20), N_theta 500, five eigenvalues kept), and the spectrum with Q fixed
(`plot_exclude_Q_spectra`, N_theta 500, sigma 1.4211, beta 2.7659).

In [ ]:
prof_ok = []
for sc in (1, 2, 4):
    for g in (1.5, 3, 5, 10):
        Jp = D.DMFTModel(N_theta=1000, g=g, scale=sc).J
        prof_ok.append(dict(sigma=sc, beta=g, J0=Jp[0], Jmin=Jp.min(), argmin_rad=2 * np.pi * min(Jp.argmin(), 1000 - Jp.argmin()) / 1000, mexican=Jp[0] > 0 and Jp.min() < 0))
print(pd.DataFrame(prof_ok).round(4).to_string(index=False))
t = time.time()
g_vals = np.logspace(-2, 2, 40); s_vals = np.logspace(0, 2, 20)
bump = np.zeros((40, 20, 5), complex); homo = np.zeros((40, 20, 5), complex); amp = np.zeros((40, 20))
for i, g in enumerate(g_vals):
    for j, sc in enumerate(s_vals):
        mdl = D.DMFTModel(N_theta=500, scale=sc, g=g)
        bump[i, j] = mdl.compute_bump_jacobian()[1][:5]
        m_b = mdl.F(mdl.Gamma_x[0], mdl.Gamma_x[0], mdl.Gamma_x)   # the bump the Jacobian is taken at
        amp[i, j] = m_b.max() - m_b.min()
        homo[i, j] = mdl.compute_homogeneous_jacobian()[1][:5]
print(f"stability grid in {time.time()-t:.0f} s")
print(f"bump eigenvalue 1 (symmetry zero): max |Re| {np.abs(bump[..., 0].real).max():.2e}; grid points where it is above 1e-6: {int((bump[..., 0].real > 1e-6).sum())} of 800")
bad = np.argwhere(bump[..., 0].real > 1e-6)
if len(bad):
    print("   at (beta, sigma):", [(round(g_vals[a], 3), round(s_vals[b], 2)) for a, b in bad[:10]], "...; there eigenvalues 1 and 2 are", [bump[a, b, :2].real.round(4).tolist() for a, b in bad[:3]])
print(f"grid points where bump eigenvalue 1 is below -1e-6 (no zero mode found): {int((bump[..., 0].real < -1e-6).sum())} of 800;",
      "at (beta, sigma):", [(round(g_vals[a], 3), round(s_vals[b], 2)) for a, b in np.argwhere(bump[..., 0].real < -1e-6)[:8]])
nz = bump[..., 0].real < -1e-6
print(f"bump amplitude (max - min of m): at the {int(nz.sum())} points without a zero mode median {np.median(amp[nz]):.3e}, range {amp[nz].min():.3e} to {amp[nz].max():.3e}; "
      f"at the other {int((~nz).sum())} median {np.median(amp[~nz]):.3e}, range {amp[~nz].min():.3e} to {amp[~nz].max():.3e}")
print(f"points with amplitude below 1e-3: {int((amp < 1e-3).sum())}, of which without a zero mode {int((nz & (amp < 1e-3)).sum())}; "
      f"largest bump eigenvalue 1 where the amplitude is at least 1e-3: {bump[..., 0].real[amp >= 1e-3].min():.3e} (most negative)")
print("bump eigenvalue 3 (first after the zero and eigenvalue 2): Re max", f"{bump[..., 2].real.max():.4e}")
print(f"bump eigenvalue 2: Re max {bump[..., 1].real.max():.4e}, negative at every grid point: {bool((bump[..., 1].real < 0).all())}")
print(f"uniform state eigenvalue 1: Re min {homo[..., 0].real.min():.4e}, positive at every point: {bool((homo[..., 0].real > 0).all())}; max |Im| {np.abs(homo[..., 0].imag).max():.2e}")
# where on the grid the bump has no zero mode: counts by beta band and by sigma band
bb = [("beta < 0.1", g_vals < 0.1), ("0.1 <= beta < 1", (g_vals >= 0.1) & (g_vals < 1)), ("beta >= 1", g_vals >= 1)]
sb = [("sigma < 10", s_vals < 10), ("10 <= sigma < 50", (s_vals >= 10) & (s_vals < 50)), ("sigma >= 50", s_vals >= 50)]
tab_nz = pd.DataFrame([[int(nz[np.ix_(bm, sm)].sum()) for _, sm in sb] + [int(nz[bm].sum())] for _, bm in bb],
                      index=[b for b, _ in bb], columns=[s for s, _ in sb] + ["all sigma"])
tab_nz.loc["all beta"] = tab_nz.sum()
print("points without a zero mode, by beta band (rows) and sigma band (columns):"); print(tab_nz.to_string())
print("grid points per band: beta", [int(bm.sum()) for _, bm in bb], "x sigma", [int(sm.sum()) for _, sm in sb])
print(f"without a zero mode with beta below 0.1 or sigma at least 50: {int((nz & (g_vals[:, None] < 0.1) | nz & (s_vals[None, :] >= 50)).sum())} of {int(nz.sum())}")
mq = D.DMFTModel(N_theta=500, scale=1.4211, g=2.7659)
w_full = mq.compute_bump_jacobian()[1]; w_noQ = mq.compute_bump_jacobian(exclude_Q=True)[1]
print("bump spectrum (leading 5 real parts):", w_full.real[:5].round(4), "| with Q fixed:", w_noQ.real[:5].round(4), "| positive with Q fixed:", bool(w_noQ.real[0] > 1e-6))

**Result: reproduced.** All twelve interaction profiles of Fig. 7a are positive at 0 (4.9 to 38.5)
and negative at some offset (minimum -1.5 to -6.0). Over the 40 by 20 grid the bump's second
eigenvalue is negative at every point (largest -4.7e-8, effectively zero at some points) and the
uniform state's leading eigenvalue is real and positive at every point (smallest 4.7e-5). With Q
fixed the bump at sigma 1.4211 and beta 2.7659 has a positive eigenvalue (0.446). At 293 of the 800
grid points no zero eigenvalue is found at the bump (the leading one is below -1e-6, down to
-0.22). Most of them are at beta of 1 and above (249 of 293) and at sigma of 10 and above (128 at
sigma 10 to 50 and 98 at 50 and above); beta below 0.1 or sigma of 50 and above holds only 99 of
them, and 67 are at sigma below 10 with beta of 1 and above. At sigma 1.4211 and beta 2.7659 the
zero is present. The bump at those points is not flat (amplitude, max - min of m, median 0.95,
below 1e-3 at only 6 of them), so its missing zero mode is not explained by a vanishing bump. The
criterion tests only the second eigenvalue; the statement's "except the symmetry-induced zero"
does not hold at those 293 points.

## fig7de-dmft-timescales

> In the mean-field theory, the convergence time to the bump manifold (-tau over the bump
> Jacobian's second eigenvalue) and the divergence time from the uniform state (tau over its
> leading eigenvalue) are both a few tau for beta of about 1 and above, decreasing with beta and
> insensitive to sigma.

In [ ]:
t_bump = -1 / bump[..., 1].real      # in units of tau
t_homo = 1 / homo[..., 0].real
sel = g_vals >= 1
sel10 = sel & (g_vals <= 10)
print("beta values with beta >= 1:", g_vals[sel].round(3))
for lab, tt in (("convergence (bump)", t_bump), ("divergence (uniform)", t_homo)):
    few = bool(((tt[sel10] >= 0.5) & (tt[sel10] <= 10)).all())
    incs = [int((np.diff(tt[sel, j]) > 0).sum()) for j in range(20)]
    dec = all(n <= 1 for n in incs)
    ratio = tt[sel].max(1) / tt[sel].min(1)
    print(f"{lab}: t*/tau over beta 1 to 10 from {tt[sel10].min():.3f} to {tt[sel10].max():.3f}, few tau (0.5 to 10): {few}; "
          f"increases along beta per sigma {incs}, decreasing: {dec}; max/min over sigma per beta {ratio.round(3)}, "
          f"max {ratio.max():.3f}: (a) <= 2 {bool(ratio.max() <= 2)}, (b) <= 3 {bool(ratio.max() <= 3)}")
    print(f"   t*/tau at beta 1.0 to 100 for sigma 1 and 100:", tt[sel][:, 0].round(3), tt[sel][:, -1].round(3))# where the extremes are, and the parts of the criterion within sigma 1 to 10
for lab, tt in (("convergence (bump)", t_bump), ("divergence (uniform)", t_homo)):
    a, b = np.unravel_index(np.argmax(np.where(sel10[:, None], tt, -np.inf)), tt.shape)
    a2, b2 = np.unravel_index(np.argmin(np.where(sel10[:, None], tt, np.inf)), tt.shape)
    print(f"{lab}: over beta 1 to 10 the largest {tt[a, b]:.3f} at beta {g_vals[a]:.3f}, sigma {s_vals[b]:.2f}; the smallest {tt[a2, b2]:.3f} at beta {g_vals[a2]:.3f}, sigma {s_vals[b2]:.2f}")
    print("   per sigma:", pd.DataFrame(dict(sigma=s_vals.round(2), t_min=tt[sel10].min(0).round(3), t_max=tt[sel10].max(0).round(3),
          few_tau=((tt[sel10] >= 0.5) & (tt[sel10] <= 10)).all(0), increases=[int((np.diff(tt[sel, j]) > 0).sum()) for j in range(20)])).to_string(index=False), sep="\n")
    s10 = s_vals <= 10
    r10 = tt[sel][:, s10].max(1) / tt[sel][:, s10].min(1)
    print(f"   over sigma 1 to 10 only: few tau {bool(((tt[np.ix_(sel10, s10)] >= 0.5) & (tt[np.ix_(sel10, s10)] <= 10)).all())}, "
          f"decreasing at every sigma {all(int((np.diff(tt[sel, j]) > 0).sum()) <= 1 for j in np.where(s10)[0])}, max/min over sigma {r10.max():.3f}")


**Result: not reproduced.** The convergence time falls with beta for every sigma and varies across
sigma by at most a factor of 2.55 at each beta (reading b holds, a does not), but over beta 1 to 10
it runs from 2.87 to 20.5 tau, the largest at beta 1.1 and sigma 37.9 (19.5 at sigma 100); it stays
within 0.5 to 10 tau only at sigma 1 and exceeds 10 tau at beta 1.1 for every sigma from 1.27 up.
The divergence time ranges from 0.15 to 3.83 tau over beta 1 to 10 and falls below 0.5 tau at
every sigma from 6.95 up; it rises with beta at the five smallest sigma (11, 10, 8, 6 and 3
increases, sigma 1 to 2.64), and varies by up to a factor of 24 across sigma (6.1 within sigma 1
to 10). No sigma meets all three parts for both timescales, and restricting the grid to sigma 1 to
10 does not change the outcome. Neither the first two parts nor the third hold for both
timescales. The thresholds are the claims file's.

## fig7c-dmft-integrates-behavior

> Under angular velocity input from mouse behaviour, the mean-field traveling-bump solutions
> track head direction by integrating angular velocity, matching the network simulations and the
> recordings.

As `dmft.ipynb` `run_sim`: N_theta 1,000, sigma 1.4211, beta 2.7659, T 110 s, step 1 ms, saved
every 10 ms, tau 50 ms, omega of A3707 interpolated linearly; the target is the cumulative sum
of omega at the save times times 10 ms; the first 20 s are dropped. The angle of the peak of m is
compared with the target, as in fig2jk-velocity-integration.

In [ ]:
om_interp = interp1d(np.arange(len(omega_data)) * 0.01, omega_data)
t = time.time()
mdl = D.DMFTModel(N_theta=1000, scale=1.4211, g=2.7659)
tt, m, m_hat, g_dyn = mdl.simulate(T=110, dt=0.001, T_eval=0.01, omega_func=om_interp, tau=0.05)
print(f"simulated in {time.time()-t:.0f} s; m {m.shape}, range {m.min():.3f} to {m.max():.3f}")
target = np.mod(np.cumsum(om_interp(tt)) * (tt[1] - tt[0]), 2 * np.pi)
keep = tt >= 20
ang = 2 * np.pi * m.argmax(1) / 1000
e = H.circ_diff(ang[keep], target[keep])
rc = circcorrcoef(ang[keep], target[keep])
print(f"{keep.sum()} samples from 20 s; median |peak angle - target| {np.median(e):.4f} rad (pi/8 {np.pi/8:.3f}); 90th percentile {np.percentile(e, 90):.4f}; circular r {rc:.4f}")
print(f"bump contrast (max - min of m) at 20 s and at the end: {np.ptp(m[keep][0]):.3f}, {np.ptp(m[-1]):.3f}")
# reference: the target shifted by 30 s
sh = 3000
print(f"reference, target shifted by 30 s: median error {np.median(H.circ_diff(ang[keep][:-sh], target[keep][sh:])):.3f} rad")

**Result: reproduced.** Driven by A3707's angular velocity, the peak of the mean-field bump
follows the integrated angular velocity with a median absolute error of 0.032 rad and circular
correlation 0.998 over 20 to 110 s, against 1.359 rad with the target shifted by 30 s. The bump
keeps its contrast (0.749 at 20 s, 0.728 at the end).

## fig6cd-eigenvalues-real-degenerate

> For tuning curves from the generative process at large N, the optimal weight matrix has real,
> doubly degenerate eigenvalues, the ratios of the input-current to the firing-rate Fourier
> coefficients scaled by the average activation slope, and they match numerical
> diagonalization at N = 2,500, with the ninth eigenvalue the unpaired constant mode.

As `spectra.ipynb` cells 3 to 7: the mean-field eigenvalues alpha Gamma_x_n / (Gamma_phi_n +
1e-12), from 50,000 idealized curves (error-function activation, sigma 1.4211, beta 2.7659) on
2,500 angles (accumulated over chunks of 5,000 to bound memory), with alpha the mean activation
slope; and the finite network, N 2,500 idealized curves on 500 angles, J by `compute_weights`
with lambda 1e-8, eigenvalues by np.linalg.eigvals. The draws come from the claim's seed (the
notebook uses numpy seeds 42 and 45).

In [ ]:
BETA, SCALE = 2.7659, 1.4211
def unity_erf(x): return erf(np.sqrt(np.pi) * x / 2.0)
def gen_idealized(n, n_theta, rng):
    """ring/tc.py gen_tuning_curves_idealized with sample_tcs, from a Generator."""
    gft = H.wrapped_gaussian_ft(SCALE, n_theta)
    nf = len(gft)
    Z = (rng.standard_normal((nf, n)) + 1j * rng.standard_normal((nf, n))) / np.sqrt(2.0)
    Z[0] = Z[0].real * np.sqrt(2.0)
    Xg = np.fft.irfft(Z * np.sqrt(gft)[:, None], axis=0, norm="forward")
    return Xg, 1 + unity_erf(BETA * Xg)
def theory_eigs(rng, n_samples=50000, n_theta=2500, chunk=5000):
    gx = np.zeros(n_theta); gp = np.zeros(n_theta); slope = 0.0
    for c0 in range(0, n_samples, chunk):
        Xg, Pg = gen_idealized(chunk, n_theta, rng)
        gx += (np.abs(np.fft.fft(Xg, axis=0, norm="forward")) ** 2).sum(1)
        gp += (np.abs(np.fft.fft(Pg, axis=0, norm="forward")) ** 2).sum(1)
        slope += (BETA * np.exp(-np.pi * (BETA * Xg) ** 2 / 4)).sum()
    gx /= n_samples; gp /= n_samples
    alpha = slope / (n_samples * n_theta)
    return alpha * gx / (gp + 1e-12), alpha
def finite_J(rng, n=2500, n_theta=500):
    Xg, Pg = gen_idealized(n, n_theta, rng)
    Jg = H.compute_J(Xg, Pg, 1e-8)
    o = np.argsort(H.circular_center_of_mass(Pg.T))
    return Jg[o][:, o], Xg[:, o], Pg[:, o]
sd0 = H.seed("fig6cd-eigenvalues-real-degenerate")
t = time.time()
lam_th, alpha = theory_eigs(np.random.default_rng(sd0 + 1))
th_sorted = np.sort(lam_th)[::-1]
n_idx = np.arange(2500)
print(f"alpha {alpha:.4f}; theory eigenvalues computed in {time.time()-t:.0f} s")
print("theory, leading 11:", th_sorted[:11].round(4), "| Fourier index of each:", [int(min(k, 2500 - k)) for k in np.argsort(lam_th)[::-1][:11]])
def judge_6cd(J6):
    w6 = np.linalg.eigvals(J6)
    w6 = w6[np.argsort(w6.real)[::-1]]
    lead = w6[:11]
    rel = np.abs(lead.real - th_sorted[:11]) / np.abs(th_sorted[:11])
    imag_ok = bool(np.all(np.abs(lead.imag) < 0.1 * np.abs(lead.real)))
    return w6, lead, rel, imag_ok
Js6 = {}
for k in range(5):
    sd = sd0 + 1000 * k
    t = time.time()
    J6, X6, P6 = finite_J(np.random.default_rng(sd))
    w6, lead, rel, imag_ok = judge_6cd(J6)
    within8 = [lead.real[2 * p] - lead.real[2 * p + 1] for p in range(4)]
    between8 = [lead.real[2 * p + 1] - lead.real[2 * p + 2] for p in range(4)]
    print(f"seed {sd} ({time.time()-t:.0f} s): leading 11 real parts {lead.real.round(4)}; relative difference from theory max over the 11 {rel.max():.4f} "
          f"(within 10%: {bool((rel <= 0.1).all())}); |Im| < 0.1 Re: {imag_ok}; gaps within pairs 1-4 {np.round(within8, 4)}, to the next {np.round(between8, 4)}; "
          f"ninth eigenvalue {lead.real[8]:.4f}, tenth {lead.real[9]:.4f}")
    Js6[k] = J6

**Result: reproduced.** Under the claim's seed the leading 11 eigenvalues at N 2,500 are within
5.0% of the mean-field values alpha Gamma_x_n / Gamma_phi_n (alpha 0.767), their imaginary parts
are under 10% of their real parts, the first four pairs are closer within than to the next pair,
and the ninth (0.181 against 0.182, Fourier index 0) is the unpaired constant mode. Under one of
the four other seeds the largest difference is 13.6%, so the outcome is seed sensitive.

## fig6fg-disordered-eigenvectors

> Spectral embedding finds rings in a circulant matrix with weak noise, whose eigenvectors are
> Fourier modes, but not in the model, whose eigenvector point clouds are structureless, while
> eigenvalue degeneracy is present in every case, so degeneracy is proposed as a robust signature
> of ring-attractor structure.

As `spectra.ipynb` cell 7: the model matrix of fig6cd (neurons sorted by center of mass) and
the circulant matrix alpha circulant(J_dmft) / N plus Gaussian noise of standard deviation
0.1 / sqrt(N). The leading singular vectors of each (eigenvectors of J J^T) are grouped into
consecutive pairs, skipping a singular vector whose overlap with the uniform vector exceeds 0.5
(the constant mode; none of the leading five does here); each neuron's point in a pair's plane has
a radius, and the coefficient of variation of the radii decides ring (below 0.2) or
structureless (above 0.4). The doublets are tested on the eigenvalues, as in fig6cd.

In [ ]:
J_dmft = np.fft.ifft(lam_th / alpha, norm="forward").real    # spectra.ipynb, compute_eff_weights: J = ifft(J_ft)
def pair_cvs(M, n_pairs=2):
    U, sv, Vt = np.linalg.svd(M)
    k = 2 * n_pairs + 1
    uni = np.abs(U[:, :k].sum(0)) / np.sqrt(M.shape[0])     # overlap with the uniform unit vector
    keep = [i for i in range(k) if uni[i] < 0.5][:2 * n_pairs]   # consecutive pairs, skipping a constant mode
    cvs = []
    for p in range(n_pairs):
        a_, b_ = U[:, keep[2 * p]], U[:, keep[2 * p + 1]]
        rad = np.hypot(a_, b_)
        cvs.append(rad.std() / rad.mean())
    return np.array(cvs), sv[:k], [i for i in range(k) if i not in keep]
def doublets(M):
    w = np.linalg.eigvals(M); w = np.sort(w.real)[::-1][:11]
    # set aside the unpaired eigenvalue: the one farthest from both neighbours among the leading 9
    d = np.array([min(abs(w[i] - w[i - 1]) if i > 0 else np.inf, abs(w[i] - w[i + 1])) for i in range(9)])
    u = int(np.argmax(d)); r = np.delete(w, u)
    within = [r[2 * p] - r[2 * p + 1] for p in range(4)]; nxt = [r[2 * p + 1] - r[2 * p + 2] for p in range(4)]
    return sum(a_ < b_ for a_, b_ in zip(within, nxt)), u
for k in range(5):
    sd = sd0 + 1000 * k
    rng = np.random.default_rng(sd + 2)   # third generator of the claim: the circulant's noise
    Jm = Js6[k]
    Jc = alpha * scipy.linalg.circulant(J_dmft) / 2500 + rng.standard_normal((2500, 2500)) * 0.1 / np.sqrt(2500)
    cv_m, sv_m, drop_m = pair_cvs(Jm); cv_c, sv_c, drop_c = pair_cvs(Jc)
    dm, um = doublets(Jm); dc, uc = doublets(Jc)
    print(f"seed {sd}: model pair CVs {cv_m.round(3)} (unused {[i+1 for i in drop_m]}); circulant+noise pair CVs {cv_c.round(3)} (unused {[i+1 for i in drop_c]}); "
          f"doublets in 4 leading pairs: model {dm}/4 (unpaired at {um+1}), circulant {dc}/4 (unpaired at {uc+1})")
    print(f"   singular values model {sv_m.round(3)}, circulant {sv_c.round(3)}")

**Result: reproduced.** For the circulant matrix plus weak noise the radii in the planes of the
two leading singular-vector pairs have coefficients of variation 0.063 and 0.070 (rings, below
0.2); for the model they are 0.53 and 0.52 (structureless, above 0.4). Both spectra show four of
four leading doublets with the unpaired eigenvalue ninth. The same holds under all five seeds.

## Summary

| claim | outcome |
|---|---|
| fig7-dmft-mexican-hat-and-stability | reproduced |
| fig7de-dmft-timescales | not reproduced |
| fig7c-dmft-integrates-behavior | reproduced |
| fig6cd-eigenvalues-real-degenerate | reproduced (seed sensitive) |
| fig6fg-disordered-eigenvectors | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

Differences from the paper's method. The paper's `ring/dmft.py` is run as it is, without
torch; the idealized tuning curves are drawn from each claim's seed (the paper's notebook uses
numpy seeds 42 and 45), and the 50,000-curve spectrum is accumulated in chunks.

In [ ]:
print(f"total time {time.time()-T0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.2f} GB")